In [1]:
!pip install -q pyspark

In [3]:
import time
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, StandardScaler, VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

Path("outputs").mkdir(exist_ok=True)
SEED = 42

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BFT65B-Part5-MLlib")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

NUM_FEATURES = [
    "LOG_ASSET", "EQ_ASSET", "RBC1AAJ", "RBCRWAJ",
    "NPL_LOANS", "NONACCR_ASSET", "PD90_ASSET", "ORE_ASSET", "RES_LOANS",
    "ROA", "ROE", "NIMY", "ROA_Q_ANN", "NCO_RATE_Q", "PROV_RATE_Q", "EFFICIENCY",
    "LIQ_ASSET", "DEP_ASSET", "BRO_ASSET", "LOANS_ASSET", "RE_LOANS",
    "ASSET_G4", "DEP_G4", "EQ_ASSET_D4", "NPL_LOANS_D4", "ROA_D4", "LOANS_ASSET_D4",
    "ROA_MA4", "ROA_SD4",
]
BASE_MISS = ["P9ASSET_MISS", "NAASSET_MISS", "NTLNLS_MISS", "ELNATR_MISS", "ROE_MISS", "BRO_MISS"]
EXTRA_SRC = ["ASSET_G4", "NPL_LOANS_D4", "ROA_MA4", "EFFICIENCY"]

df = spark.read.parquet("features.parquet")
for c in EXTRA_SRC:
    df = df.withColumn(f"{c}_MISS", F.when(F.col(c).isNull(), F.lit(1.0)).otherwise(F.lit(0.0)))
for c in BASE_MISS:
    df = df.withColumn(c, F.col(c).cast("double"))

ALL_MISS = BASE_MISS + [f"{c}_MISS" for c in EXTRA_SRC]
df = df.withColumn("label", F.col("y").cast("double"))

# Temporal split inside DEV matching Fold 3 (4-quarter purge gap: 2007 excluded)
# Fit strictly on <= 2006-12-31, evaluate on 2008-03-31 .. 2009-12-31 (Test set never touched!)
train_df = df.filter(F.col("QDATE") <= F.lit("2006-12-31").cast("date")).cache()
val_df = df.filter(
    (F.col("QDATE") >= F.lit("2008-03-31").cast("date"))
    & (F.col("QDATE") <= F.lit("2009-12-31").cast("date"))
).cache()

n_tr, n_va = train_df.count(), val_df.count()

# Winsorize thresholds learned strictly on train_df via approxQuantile (relative error 0.001)
bounds = train_df.stat.approxQuantile(NUM_FEATURES, [0.005, 0.995], 0.001)
for col_name, (lo, hi) in zip(NUM_FEATURES, bounds):
    train_df = train_df.withColumn(
        col_name,
        F.when(F.col(col_name).isNull(), None)
         .when(F.col(col_name) < lo, F.lit(float(lo)))
         .when(F.col(col_name) > hi, F.lit(float(hi)))
         .otherwise(F.col(col_name)),
    )
    val_df = val_df.withColumn(
        col_name,
        F.when(F.col(col_name).isNull(), None)
         .when(F.col(col_name) < lo, F.lit(float(lo)))
         .when(F.col(col_name) > hi, F.lit(float(hi)))
         .otherwise(F.col(col_name)),
    )

imp_cols = [f"{c}_IMP" for c in NUM_FEATURES]
imputer = Imputer(strategy="median", inputCols=NUM_FEATURES, outputCols=imp_cols)
num_asm = VectorAssembler(inputCols=imp_cols, outputCol="num_vec")
scaler = StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=True, withStd=True)
full_asm = VectorAssembler(inputCols=["num_scaled"] + ALL_MISS, outputCol="features")

eval_pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

lines = [
    f"Spark MLlib Demo (Spark {spark.version}) on DEV Fold-3 (Train<=2006Q4: {n_tr:,} rows | Val 2008Q1-2009Q4: {n_va:,} rows)",
]

models = {
    "MLlib_LogisticRegression_L2": LogisticRegression(featuresCol="features", labelCol="label", regParam=0.01, maxIter=100),
    "MLlib_GBTClassifier": GBTClassifier(featuresCol="features", labelCol="label", maxIter=40, maxDepth=5, seed=SEED),
}

for name, clf in models.items():
    t0 = time.perf_counter()
    pipe = Pipeline(stages=[imputer, num_asm, scaler, full_asm, clf])
    fitted = pipe.fit(train_df)
    preds = fitted.transform(val_df)
    pr_auc = eval_pr.evaluate(preds)
    roc_auc = eval_roc.evaluate(preds)
    dt = time.perf_counter() - t0
    lines.append(f"  {name:<28} -> PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f} | fit+eval={dt:.2f}s")

report = "\n".join(lines)
print(report)
Path("outputs/part5_mllib_report.txt").write_text(report + "\n", encoding="utf-8")

from google.colab import files
files.download("outputs/part5_mllib_report.txt")

Spark MLlib Demo (Spark 4.0.5) on DEV Fold-3 (Train<=2006Q4: 1,193,676 rows | Val 2008Q1-2009Q4: 66,919 rows)
  MLlib_LogisticRegression_L2  -> PR-AUC=0.5623 | ROC-AUC=0.9681 | fit+eval=124.23s
  MLlib_GBTClassifier          -> PR-AUC=0.6378 | ROC-AUC=0.9682 | fit+eval=284.41s


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>